In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mahotas as mh
import imutils
import nd2
import cv2
import os
from scipy.signal import find_peaks
pwd = os.getcwd()

In [ ]:
IMAGE_SCALE = 0.5605 # pixel/micron
SCALE_BAR = 200 # micron
SCALE_BAR_MARGIN = 50 # pixels
SCALE_BAR_THICKNESS = 20 # pixels

path_results = ['./cropped_images']
folder_address = ['./raw_images']
# path_results = os.path.join(pwd,results_address[0] + '_results', 'sliced_images')
if not os.path.exists(path_results[0]):
    os.makedirs(path_results[0])

In [ ]:
# for .tif images
all_imgs_names = [f for f in os.listdir(folder_address[0]) \
              if (os.path.splitext(os.path.join(pwd,f))[1] == '.tif')]

# for .jpg images
#all_imgs_names = [f for f in os.listdir(folder_address[0]) \
#                   if (os.path.splitext(os.path.join(pwd,f))[1] == '.jpg')]

print(len(all_imgs_names))

In [ ]:
for img_num in range(len(all_imgs_names)):
    print(all_imgs_names[img_num])
    
    # Read the color image
    img_color = cv2.imread(os.path.join(folder_address[0], 
                                        all_imgs_names[img_num]), 
                           cv2.IMREAD_UNCHANGED)
    
    # Convert color image to grayscale
    # bf_img = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY) # chatgpt
    bf_img = cv2.imread(os.path.join(folder_address[0],
                                     all_imgs_names[img_num]), 
                            cv2.IMREAD_GRAYSCALE)
    
    # print(np.max(bf_img))
    bf_img = bf_img * int(255/np.max(bf_img))
    plt.figure()
    plt.imshow(bf_img, 'gray')
    plt.show()

    histogram, bins = np.histogram(bf_img.ravel(), 256, [0,256])
    w = 4
    histogram = np.convolve(histogram, np.ones(w), 'valid') / w
    # print(histogram)
    plt.figure()
    plt.plot(histogram)
    plt.show()
         
    
    # for local maxima
    #bin_mean = 140 #random custom number that will white out the background
    maxima_indices = find_peaks(histogram, distance= 50)
    # print("maxima_indices :", maxima_indices)
    bin_max = np.max(maxima_indices[0])
    bin_min = np.min(maxima_indices[0])
    bin_mean = np.mean([bin_min, bin_max])
    # print("bin_max :", bin_max)
    # print("bin_min :", bin_min)

    
    
    ret, mask_img = cv2.threshold(bf_img, bin_mean, 255, cv2.THRESH_BINARY_INV)
    
    plt.figure()
    plt.imshow(mask_img, 'gray')
    plt.show()
    
    labeled_img, n_img = mh.label(mask_img)
    labeled_img, n_img = mh.labeled.filter_labeled(labeled_img, 
                                                   remove_bordering=True, 
                                                   min_size=10000)
    plt.figure()
    plt.imshow(labeled_img, 'gray')
    plt.show()
    nn = 1
    
    for obj in np.unique(labeled_img):
        # if the label is zero, we are examining the 'background', so simply ignore it
        if obj == 0:
            continue
            
        # otherwise, allocate memory for the label region and draw it on the mask
        mask = np.zeros(labeled_img.shape, dtype="uint8")
        mask[labeled_img == obj] = 255

        # detect contours in the green mask and grab the largest one
        cnts = cv2.findContours(mask.copy(), 
                                cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
        cnts = imutils.grab_contours(cnts)
        c = max(cnts, key=cv2.contourArea)

        
        # draw a circle enclosing the object
        ((x_obj, y_obj), r_obj) = cv2.minEnclosingCircle(c)
        cv2.circle(img_color, (int(x_obj), int(y_obj)), int(r_obj), 255, 2)
        plt.figure()
        plt.imshow(img_color, 'gray')
        plt.show()
        
        CROP_W_SIZE = int(r_obj*2*1.25) 
        CROP_H_SIZE = int(r_obj*2*1.25) 
        
        # Calculate cropping parameters
        x = int(x_obj - CROP_H_SIZE/2)
        y = int(y_obj - CROP_W_SIZE/2)
        h = CROP_H_SIZE
        w = CROP_W_SIZE

        # Perform boundary checks
        x = max(0, x)
        y = max(0, y)
        h = min(h, bf_img.shape[0] - y)
        w = min(w, bf_img.shape[1] - x)

        # Crop the image
        img_2 = bf_img[y:y+h, x:x+w]        

        
        # Adjust SCALE_BAR_LENGTH based on the actual size of the cropped image
        scaled_bar_length = int (SCALE_BAR * IMAGE_SCALE) 
        scaled_bar_margin = int(SCALE_BAR_MARGIN * (w / CROP_W_SIZE))
        
        # Adjust start_point based on the actual size of the cropped image
        start_point = (w - scaled_bar_margin - scaled_bar_length, h - SCALE_BAR_MARGIN - SCALE_BAR_THICKNESS)
        
        # Draw the rectangle on the cropped image
        end_point = (w - scaled_bar_margin, h - SCALE_BAR_MARGIN)
        img_2 = cv2.rectangle(img_2, start_point, end_point, 0, -1)
      
    
        # Not adjusted SCALE_BAR_LENGTH:
        # SCALE_BAR_LENGTH = int(SCALE_BAR * IMAGE_SCALE)
        # start_point = (CROP_H_SIZE-SCALE_BAR_MARGIN-SCALE_BAR_LENGTH, CROP_W_SIZE-SCALE_BAR_MARGIN-SCALE_BAR_THICKNESS)
        # end_point = (CROP_H_SIZE-SCALE_BAR_MARGIN, CROP_W_SIZE-SCALE_BAR_MARGIN)
        # img_2 = cv2.rectangle(img_2, start_point, end_point, 0, -1)
    
    
        # Display the result
        plt.figure()
        plt.imshow(img_2, 'gray')
        plt.show()
        
        # Save the cropped image
        cv2.imwrite(os.path.join(path_results[0], os.path.splitext(os.path.basename(os.path.join(folder_address[0], 
                                                                                                 all_imgs_names[img_num])))[0]+'_'+str(nn)+'.tif'), img_2)
        
        nn += 1